# Audio Appendix: EDA and Model Evidence

This appendix expands the audio portion of the final prototype evidence notebook. The master reference remains `notebooks/00_final_prototype_evidence_notebook.ipynb`.

Purpose: document ASVspoof audio readiness, train/dev balance, MFCC/SVM metrics, behavior-model evidence, trained voice-evidence calibration, and runtime traceability without rerunning audio training.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def project_path(relative: str) -> Path:
    return ROOT / relative

def load_json(relative: str) -> dict:
    path = project_path(relative)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def status(path: Path) -> str:
    return "PASS" if path.exists() else "MISSING"

ROOT

WindowsPath('C:/Users/user/Documents/Codex/ai-scam-detector')

## Dashboard Input, Visible Output, and Report Handoff

This appendix starts from the audio evidence a user can see. The key point is that raw audio is not useful in a text report by itself, so the dashboard converts it into derived evidence: transcript text, audio concern signals, model outputs, flags, and explanations.

In [2]:
dashboard_io_rows = [
    {
        "Dashboard step": "Provide audio evidence",
        "User-visible input": "Upload or record WAV, MP3, M4A, FLAC, or another supported audio format.",
        "User-visible output": "Uploaded/recorded audio item, transcript preview when available, and analysis controls.",
        "Report/history fields": "source_name where available, preview/transcript text where generated",
        "Documentation note": "Reports should describe derived audio evidence, not depend on raw audio playback.",
    },
    {
        "Dashboard step": "Transcribe and analyse",
        "User-visible input": "Choose Whisper/transcript settings and run analysis.",
        "User-visible output": "Transcription result, transcript scam result, audio/MFCC concern, behavior evidence, and speech-quality notes where available.",
        "Report/history fields": "prediction, confidence, model/model_name, preview, flags, explanation",
        "Documentation note": "Audio becomes reportable after it is summarized into text and model evidence.",
    },
    {
        "Dashboard step": "Review visible evidence",
        "User-visible input": "No extra input after analysis.",
        "User-visible output": "Voice/authenticity concern score, transcript-risk signal, model confidence, flags, charts, and explanation.",
        "Report/history fields": "Stored summary fields vary by workflow; cite visible dashboard evidence separately when needed.",
        "Documentation note": "Keep acoustic evidence, transcript evidence, and report storage boundaries separate.",
    },
    {
        "Dashboard step": "Convert to report evidence",
        "User-visible input": "Open AI Report Generator and select saved audio/transcript evidence rows.",
        "User-visible output": "Report preview and TXT/PDF/DOCX download.",
        "Report/history fields": "scan_type, prediction, confidence, model_name, preview, flags, explanation, raw_input where available",
        "Documentation note": "This makes the report readable even when the original evidence was audio.",
    },
]

pd.DataFrame(dashboard_io_rows)

,Dashboard step,User-visible input,User-visible output,Report/history fields,Documentation note
0,Provide audio evidence,"Upload or record WAV, MP3, M4A, FLAC, or anoth...","Uploaded/recorded audio item, transcript previ...","source_name where available, preview/transcrip...",Reports should describe derived audio evidence...
1,Transcribe and analyse,Choose Whisper/transcript settings and run ana...,"Transcription result, transcript scam result, ...","prediction, confidence, model/model_name, prev...",Audio becomes reportable after it is summarize...
2,Review visible evidence,No extra input after analysis.,"Voice/authenticity concern score, transcript-r...",Stored summary fields vary by workflow; cite v...,"Keep acoustic evidence, transcript evidence, a..."
3,Convert to report evidence,Open AI Report Generator and select saved audi...,Report preview and TXT/PDF/DOCX download.,"scan_type, prediction, confidence, model_name,...",This makes the report readable even when the o...


## Dataset Readiness

Audio evidence uses ASVspoof-style bonafide/spoof data. Unlike email and transcript text, audio uses a train/dev validation style and signal features such as MFCC, spectral, energy, and timing statistics.

In [3]:
labels_path = project_path("data/processed/audio/labels.csv")
audio_labels_df = pd.read_csv(labels_path) if labels_path.exists() else pd.DataFrame()

pd.DataFrame([
    {
        "Evidence item": "Processed audio labels",
        "Path": str(labels_path.relative_to(ROOT)),
        "Status": status(labels_path),
        "Rows": len(audio_labels_df),
        "Columns": len(audio_labels_df.columns),
        "Label column present": "label" in audio_labels_df.columns,
    }
])

,Evidence item,Path,Status,Rows,Columns,Label column present
0,Processed audio labels,data\processed\audio\labels.csv,PASS,2600,7,True


In [4]:
if not audio_labels_df.empty and "label" in audio_labels_df.columns:
    counts = audio_labels_df["label"].value_counts().sort_index()
    ax = counts.plot(kind="bar", figsize=(6, 4), color=["#0F766E", "#7C3AED"])
    ax.set_title("Audio Label Distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Rows")
    plt.tight_layout()
else:
    print("Processed audio labels or label column is unavailable.")

## Audio Metric Evidence

The final audio evidence uses saved metric JSON files. The MFCC/statistical model provides the raw voice-authenticity signal, the behavior Random Forest provides secondary speech-behavior evidence where available, and the trained voice-evidence calibrator converts those upstream signals plus quality metadata into the dashboard-facing `voice_evidence_risk`.


In [5]:
audio_metrics = load_json("reports/metrics/audio_model_metrics.json")
behavior_metrics = load_json("reports/metrics/audio_behavior_metrics.json")
voice_evidence_metrics = load_json("reports/metrics/audio_voice_evidence_metrics.json")


def selected_audio_metrics(data: dict) -> dict:
    metrics = data.get("metrics", {})
    if isinstance(metrics, dict) and isinstance(metrics.get("full"), dict):
        return metrics["full"]
    if isinstance(metrics, dict) and isinstance(metrics.get("all_dev_variants"), dict):
        return metrics["all_dev_variants"]
    return metrics if isinstance(metrics, dict) else {}


def metric_value(metrics: dict, *names: str):
    for name in names:
        if metrics.get(name) is not None:
            return metrics.get(name)
    return None

rows = []
model_specs = [
    ("MFCC/statistical SVM", audio_metrics, "Raw bonafide/spoof audio signal"),
    ("Behavior Random Forest", behavior_metrics, "Secondary speech-behavior evidence"),
    ("Voice evidence calibrator", voice_evidence_metrics, "Dashboard-facing calibrated voice evidence risk"),
]

for label, data, role in model_specs:
    metrics = selected_audio_metrics(data)
    rows.append({
        "Model": label,
        "Runtime role": role,
        "Train samples": data.get("train_samples") or data.get("train_rows"),
        "Dev samples": data.get("dev_samples") or data.get("dev_rows"),
        "Feature dimension": data.get("feature_dimension"),
        "Accuracy": metric_value(metrics, "accuracy", "threshold_accuracy"),
        "Precision": metric_value(metrics, "precision", "threshold_precision"),
        "Recall": metric_value(metrics, "recall", "threshold_recall"),
        "F1": metric_value(metrics, "f1", "threshold_f1"),
        "ROC-AUC": metric_value(metrics, "roc_auc", "truth_roc_auc"),
    })

pd.DataFrame(rows)


,Model,Runtime role,Train samples,Dev samples,Feature dimension,Accuracy,Precision,Recall,F1,ROC-AUC
0,MFCC/statistical SVM,Raw bonafide/spoof audio signal,2000,600,251,0.936667,0.896970,0.986667,0.939683,0.993922
1,Behavior Random Forest,Secondary speech-behavior evidence,2000,600,13,0.821667,0.840989,0.793333,0.816467,0.878772
2,Voice evidence calibrator,Dashboard-facing calibrated voice evidence risk,110,50,30,0.760000,1.000000,0.520000,0.684211,0.947200


In [6]:
importance_df = pd.DataFrame(behavior_metrics.get("feature_importances", []))
if not importance_df.empty:
    plot_df = importance_df.head(10).sort_values("importance", ascending=True)
    ax = plot_df.plot.barh(x="feature", y="importance", figsize=(8, 5), color="#2563EB", legend=False)
    ax.set_title("Top Audio Behavior Feature Importances")
    ax.set_xlabel("Importance")
    plt.tight_layout()
else:
    print("No saved behavior feature importances found.")

## Runtime Artifact And Source Traceability

The dashboard should load saved audio artifacts and combine audio concern with transcript/speech-quality signals where available. This notebook verifies that the key evidence files exist and separates the upstream audio models from the final voice-evidence calibration layer.


In [7]:
artifact_paths = [
    "models/audio_svm.pkl",
    "models/audio_behavior_rf.pkl",
    "models/audio_voice_evidence_calibrator.pkl",
    "reports/metrics/audio_model_metrics.json",
    "reports/metrics/audio_behavior_metrics.json",
    "reports/metrics/audio_voice_evidence_metrics.json",
    "app/transcript_tab.py",
    "src/audio/live_audio_analysis.py",
    "src/audio/voice_evidence_calibrator.py",
    "src/training/audio_trainer.py",
    "src/training/audio_behavior_trainer.py",
    "src/training/audio_voice_evidence_trainer.py",
    "scripts/06_train_audio_model.py",
    "scripts/07_train_audio_behavior_model.py",
    "scripts/08_train_audio_voice_evidence_calibrator.py",
]

pd.DataFrame([
    {"Path": relative, "Status": status(project_path(relative))}
    for relative in artifact_paths
])


,Path,Status
0,models/audio_svm.pkl,PASS
1,models/audio_behavior_rf.pkl,PASS
2,models/audio_voice_evidence_calibrator.pkl,PASS
3,reports/metrics/audio_model_metrics.json,PASS
4,reports/metrics/audio_behavior_metrics.json,PASS
5,reports/metrics/audio_voice_evidence_metrics.json,PASS
6,app/transcript_tab.py,PASS
7,src/audio/live_audio_analysis.py,PASS
8,src/audio/voice_evidence_calibrator.py,PASS
9,src/training/audio_trainer.py,PASS


## Reviewer Note

Audio analysis can be affected by noise, compression, recording length, accents, overlapping speakers, and missing FFmpeg or Whisper dependencies. Treat audio output as an educational concern signal that supports human review.

For presentation, describe the final audio output as **voice evidence risk** or **voice-authenticity concern**. The trained calibrator improves dashboard calibration from raw voice and behavior scores, but it is not legal proof of cloning, identity, intent, or fraud.
